In [19]:
# carga la extensión autoreload
%load_ext autoreload

# establece el nivel de autorecarga en 2
%autoreload 2

import sys
sys.path.append('../')

from db.client import MongoDBConnection
import pandas as pd
import re

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
mongo_connection = MongoDBConnection()
db = mongo_connection.get_database()

In [21]:
collection = db["operaciones"]

In [ ]:
df_operaciones = pd.DataFrame(list(
    collection.aggregate([{"$unwind": "$acciones"}, {"$project": {
        "_id": 0,
        "archivo_origen": "$archivo_origen",
        "pagina_origen": "$pagina_origen",
        "politica": "$politica",
        "proceso": "$proceso",
        "operacion": "$operacion",
        "avance_operacion": "$avance_operacion",
        "A": "$acciones.accion",
        "B": {"$literal": 0},
        "C": "$acciones.producto",
        "D": "$acciones.meta",
        "E": {"$literal": 0},
        "F": "$acciones.trimestres.T1_porcentaje",
        "G": {"$literal": 0},
        "H": "$acciones.trimestres.T2_porcentaje",
        "I": {"$literal": 0},
        "J": "$acciones.trimestres.T3_porcentaje",
        "K": {"$literal": 0},
        "L": "$acciones.trimestres.T4_porcentaje",
        "M": "$acciones.responsable",
        "N": "N/A",
        "O": "Ninguna",
        "P": "Ninguna",
        "Q": "$acciones.trimestres.T1_porcentaje",
        "R": {"$literal": 0},
        "S": "$acciones.observaciones.T1",
        "T": {"$literal": 0},
        "U": "$acciones.trimestres.T2_porcentaje",
        "V": {"$literal": 0},
        "W": "$acciones.observaciones.T2",
        "X": {"$literal": 0},
        "Y": "$acciones.trimestres.T3_porcentaje",
        "Z": {"$literal": 0},
        "AA": "$acciones.observaciones.T3",
        "AB": {"$literal": 0},
        "AC": "$acciones.trimestres.T4_porcentaje",
        "AD": {"$literal": 0},
        "AE": "$acciones.observaciones.T4",
        "AF": {"$literal": 0}
    }}])
))

In [23]:
df_operaciones

,archivo,A,B,C,D,E,F,G,H,I,...,W,X,Y,Z,AA,AB,AC,AD,AE,AF
0,datos/Siplan- Innovación.pdf,Apoyar actividades para implementar caja de he...,0,Caja de herramientas del Código de integridad ...,100%,0,0%,0,50%,0,...,None,0,75%,0,mediante memorandos 20251VBG-OG-043356 de fech...,0,0%,0,No se encuentra ninguna observación,0
1,datos/Siplan- Innovación.pdf,Apoyar actividades para implementar políticas ...,0,Actividades orientadas al fortalecimiento de l...,100,0,0%,0,50%,0,...,None,0,75%,0,mediante memorandos 20251VBG-OG-043356 de fech...,0,0%,0,No se encuentra ninguna observación,0
2,datos/Siplan- Innovación.pdf,Apoyar actividades para implementar un plan de...,0,Plan de trabajo de cultura organizacional impl...,100,0,0%,0,50%,0,...,None,0,75%,0,El día 4 de julio de 2025 se realizó actividad...,0,0%,0,No se encuentra ninguna observación,0
3,datos/Siplan- Innovación.pdf,"Formular, publicar, actualizar y realizar el P...",0,"Plan Anual de Adquisiciones formulado, publica...",100%,0,0%,0,0%,0,...,None,0,0%,0,No se encuentra ninguna observación,0,0%,0,No se encuentra ninguna observación,0
4,datos/Siplan- Innovación.pdf,Obligar del 90.74% de los recursos de inversió...,0,Recursos de inversión obligados,90.74%,0,0%,0,0%,0,...,None,0,0%,0,No se encuentra ninguna observación,0,0%,0,No se encuentra ninguna observación,0
5,datos/Siplan- Innovación.pdf,Obligar 92.65% de los recursos de la reserva p...,0,Recursos obligados de la reserva presupuestal ...,92.65%,0,0%,0,0%,0,...,None,0,0%,0,No se encuentra ninguna observación,0,0%,0,No se encuentra ninguna observación,0
6,datos/Siplan- Innovación.pdf,Formular oportunamente el plane operativo 2025...,0,Plan operativo formulado,1,0,100%,0,100%,0,...,None,0,100%,0,Plan operativo formulado oportunamente,0,0%,0,No se encuentra,0
7,datos/Siplan- Innovación.pdf,Ejecutar las actividades definidas para la vig...,0,Modelo de seguridad y privacidad de la informa...,100%,0,10%,0,30%,0,...,None,0,60%,0,Mediante memorando 20251-VBOG-043355 se solici...,0,0%,0,No se encuentra ninguna observación,0
8,datos/Siplan- Innovación.pdf,"Atender las peticiones, quejas, reclamos y den...",0,PQRD atendidas de acuerdo a ley y demás dispos...,100%,0,25%,0,50%,0,...,None,0,75%,0,PQRD atendidas oportunamente,0,0%,0,No se encuentra ninguna observación,0
9,datos/Siplan- Innovación.pdf,Implementar la estrategia de participación ciu...,0,Estrategia de participación ciudadana aprobada...,100,0,0%,0,10%,0,...,None,0,50%,0,Se realizaron tres eventos de hablemos de inno...,0,75%,0,No se encuentra ninguna observación,0


In [24]:
p

NameError: name 'p' is not defined

In [ ]:
def separar_por_trimestre(texto: str):
    if not texto:
        return {
            "T1": None,
            "T2": None,
            "T3": None,
            "T4": None,
        }

    resultado = {
        "T1": None,
        "T2": None,
        "T3": None,
        "T4": None,
    }

    # Regex para capturar trimestre y contenido hasta el siguiente "Observa"
    pattern = r"Observa\s+(\d)°\s+Trimestre:(.*?)(?=Observa\s+\d°\s+Trimestre:|$)"

    matches = re.findall(pattern, texto, flags=re.DOTALL)

    for trimestre, contenido in matches:
        key = f"T{trimestre}"
        resultado[key] = contenido.strip()

    return resultado


In [ ]:
for doc in collection.find():
    
    nuevas_acciones = []

    for accion in doc.get("acciones", []):
        
        descripcion = accion.get("descripcion")
        observaciones = separar_por_trimestre(descripcion)

        accion["observaciones"] = observaciones
        
        # Si quieres eliminar descripcion original:
        accion.pop("descripcion", None)

        nuevas_acciones.append(accion)

    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": {"acciones": nuevas_acciones}}
    )

print("Migración completada ✅")

Migración completada ✅


In [ ]:
separar_por_trimestre("Observa 3° Trimestre:Mediante memorandos Nos. 20251-VBOG-022561 de fecha 12-4-2025; 20251-VBOG-030833 de fecha 8 mayo de 2025; 20251-VBOG-035683 de fecha 28 mayo de 2025 y 20251-VBOG-042028 de 20 junio de 2025 se solicitó capacitación a la Subdirección administrativa.Observa 4° Trimestre: No se encuentra ninguna observación.")

{'T1': None,
 'T2': None,
 'T3': 'Mediante memorandos Nos. 20251-VBOG-022561 de fecha 12-4-2025; 20251-VBOG-030833 de fecha 8 mayo de 2025; 20251-VBOG-035683 de fecha 28 mayo de 2025 y 20251-VBOG-042028 de 20 junio de 2025 se solicitó capacitación a la Subdirección administrativa.',
 'T4': 'No se encuentra ninguna observación.'}